In [3]:
import torch
import sys
from pathlib import Path
import torch, torch.nn as nn
import torchvision.models as models

# Add the project root (parent of NoteBooks) to Python's path
project_root = Path.cwd().parent  # if notebook is in NoteBooks/
sys.path.insert(0, str(project_root))



from src.training.train_cnn import create_cnn_model
from src.models.mlp import create_mlp_model_from_splits
from src.data.loader import *
from src.data.preprocessing import * # need thsi for just a sec. 
from scripts.ensemble_evalute import final_evaluate_mlp_on_test, final_evaluate_cnn_on_test
from src.models.cnn import FinetunedDenseNet
from src.models.mlp import MLP

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
projections_path = "../IU-xray/indiana_projections.csv"
reports_path = "../IU-xray/indiana_reports.csv"
image_path = '../IU-xray/images/images_normalized'

In [ ]:
"""
triage_scoring.py
Load trained models and generate triage scores for all test patients.
"""
import torch
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from datetime import datetime

# Paths
SPLITS_DIR = Path("../data/splits")
MODELS_DIR = Path("../models")
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_models_and_data(mode="pneumonia", text_amount="medium"):
    """
    Load all models and test data for a given mode.
    """
    print(f"\n{'='*60}")
    print(f"LOADING MODELS FOR: {mode.upper()}")
    print(f"{'='*60}")
    
    # Load test data
    test_df = pd.read_csv(SPLITS_DIR / f"{mode}_test.csv")
    print(f"✅ Loaded test set: {len(test_df)} patients")
    print(f"   Positives: {test_df['label'].sum()} ({test_df['label'].mean():.1%})")
    
    # Load MLP
    mlp_path = MODELS_DIR / f"best_mlp_{mode}_{text_amount}.pth"
    mlp_state = torch.load(mlp_path, map_location=device)
    input_dim = mlp_state['network.0.weight'].shape[1]
    mlp_model = MLP(input_nodes=input_dim).to(device)
    mlp_model.load_state_dict(mlp_state)
    mlp_model.eval()
    print(f"✅ Loaded MLP (input dim: {input_dim})")
    
    # Load vectorizer
    vectorizer_path = MODELS_DIR / f"vectorizer_{mode}_{text_amount}.pkl"
    with open(vectorizer_path, 'rb') as f:
        vectorizer = pickle.load(f)
    print(f"✅ Loaded vectorizer")
    
    # Load CNN
    from src.models.cnn import FinetunedDenseNet
    cnn_path = MODELS_DIR / f"best_cnn_densenet_{mode}.pth"
    cnn_state = torch.load(cnn_path, map_location=device)
    cnn_model = FinetunedDenseNet().to(device)
    cnn_model.load_state_dict(cnn_state)
    cnn_model.eval()
    print(f"✅ Loaded CNN")
    
    return test_df, mlp_model, cnn_model, vectorizer


def prepare_text_for_mlp(test_df, vectorizer, mode="medium"):
    """
    Prepare text features for MLP prediction.
    """
    from src.data.nlp_preprocessing import build_nlp_pipeline, prepare_text_features
    
    nlp = build_nlp_pipeline()
    test_df = prepare_text_features(test_df.copy(), nlp, mode=mode)
    X_test = vectorizer.transform(test_df["text_clean"]).toarray()
    
    return X_test


def get_cnn_predictions(cnn_model, test_df, image_dir, device):
    """
    Get CNN predictions for all test images.
    """
    from src.data.dataset import ChestXRayDataset
    from src.data.loader import get_transforms_grayscale
    from torch.utils.data import DataLoader
    
    _, test_transforms = get_transforms_grayscale()
    test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    cnn_probs = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            logits = cnn_model(images).squeeze()
            probs = torch.sigmoid(logits).cpu().numpy()
            if hasattr(probs, '__len__'):
                cnn_probs.extend(probs)
            else:
                cnn_probs.append(probs)
    
    return np.array(cnn_probs)


def calculate_triage_score(mlp_prob, cnn_prob, weights=(0.5, 0.5)):
    """
    Calculate weighted triage score from model probabilities.
    
    Returns:
        triage_score: 0-100 score
        risk_level: "HIGH", "MEDIUM", or "LOW"
        recommendation: Clinical recommendation
    """
    # Weighted ensemble
    ensemble_prob = weights[0] * mlp_prob + weights[1] * cnn_prob
    
    # Convert to 0-100 score
    triage_score = ensemble_prob * 100
    
    # Risk stratification
    if triage_score >= 50:
        risk_level = "HIGH"
        recommendation = "Immediate radiologist review required"
    elif triage_score >= 20:
        risk_level = "MEDIUM"
        recommendation = "Priority review within 4 hours"
    else:
        risk_level = "LOW"
        recommendation = "Routine review within 24 hours"
    
    return triage_score, risk_level, recommendation, ensemble_prob


def generate_triage_report(mode="pneumonia", image_dir=None, text_amount="medium"):
    """
    Generate complete triage report for all test patients.
    """
    # Load everything
    test_df, mlp_model, cnn_model, vectorizer = load_models_and_data(mode, text_amount)
    
    print(f"\n{'='*60}")
    print(f"GENERATING TRIAGE SCORES")
    print(f"{'='*60}")
    
    # Get MLP predictions
    print("\n🔮 Computing MLP predictions...")
    X_test = prepare_text_for_mlp(test_df, vectorizer, mode=text_amount)
    with torch.no_grad():
        X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        mlp_logits = mlp_model(X_tensor).squeeze()
        mlp_probs = torch.sigmoid(mlp_logits).cpu().numpy()
    
    # Get CNN predictions
    print("🖼️ Computing CNN predictions...")
    cnn_probs = get_cnn_predictions(cnn_model, test_df, image_dir, device)
    
    # Create results dataframe
    results = test_df[['uid', 'filename']].copy()
    results['true_label'] = test_df['label'].values
    results['mlp_prob'] = mlp_probs
    results['cnn_prob'] = cnn_probs
    
    # Calculate triage scores
    results['triage_score'], results['risk_level'], results['recommendation'], results['ensemble_prob'] = zip(*[
        calculate_triage_score(m, c) for m, c in zip(mlp_probs, cnn_probs)
    ])
    
    # Add text snippets for context (first 100 chars of indication/impression)
    results['indication'] = test_df['indication'].fillna('')[:100]
    results['impression'] = test_df['impression'].fillna('')[:100]
    
    # Sort by triage score (highest risk first)
    results = results.sort_values('triage_score', ascending=False).reset_index(drop=True)
    
    # Add rank
    results.insert(0, 'rank', range(1, len(results) + 1))
    
    print(f"\n✅ Generated triage scores for {len(results)} patients")
    
    return results


def print_triage_summary(results, mode):
    """Print summary statistics of triage scores."""
    print(f"\n{'='*60}")
    print(f"TRIAGE SUMMARY - {mode.upper()}")
    print(f"{'='*60}")
    
    # Overall stats
    print(f"\n📊 Score Distribution:")
    print(f"   Mean:   {results['triage_score'].mean():.1f}")
    print(f"   Median: {results['triage_score'].median():.1f}")
    print(f"   Min:    {results['triage_score'].min():.1f}")
    print(f"   Max:    {results['triage_score'].max():.1f}")
    
    # Risk level breakdown
    print(f"\n🚦 Risk Level Breakdown:")
    risk_counts = results['risk_level'].value_counts()
    for level, count in risk_counts.items():
        pct = count / len(results) * 100
        print(f"   {level}: {count} patients ({pct:.1f}%)")
    
    # Performance by true label
    print(f"\n📈 Performance by True Label:")
    for label, name in [(1, "Positive (Has Condition)"), (0, "Negative (No Condition)")]:
        subset = results[results['true_label'] == label]
        if len(subset) > 0:
            print(f"\n   {name}:")
            print(f"      Mean Triage Score: {subset['triage_score'].mean():.1f}")
            print(f"      High Risk: {sum(subset['risk_level'] == 'HIGH')} ({sum(subset['risk_level'] == 'HIGH')/len(subset):.1%})")
            print(f"      Medium Risk: {sum(subset['risk_level'] == 'MEDIUM')} ({sum(subset['risk_level'] == 'MEDIUM')/len(subset):.1%})")
            print(f"      Low Risk: {sum(subset['risk_level'] == 'LOW')} ({sum(subset['risk_level'] == 'LOW')/len(subset):.1%})")
    
    # Top 10 highest risk patients
    print(f"\n🔴 TOP 10 HIGHEST RISK PATIENTS:")
    print("-" * 80)
    top10 = results.head(10)
    for _, row in top10.iterrows():
        true_str = "✓" if row['true_label'] == 1 else "✗"
        
        # ✅ FIXED: Handle NaN in indication
        indication = row.get('indication', '')
        if pd.isna(indication) or indication == '':
            indication = '(no indication provided)'
        else:
            indication = str(indication)[:50]
        
        print(f"  Rank {int(row['rank']):2d}: Score {row['triage_score']:5.1f} | {row['risk_level']:6s} | True: {true_str} | {indication}...")
    
    # Bottom 10 lowest risk
    print(f"\n🟢 BOTTOM 10 LOWEST RISK PATIENTS:")
    print("-" * 80)
    bottom10 = results.tail(10).sort_values('triage_score')
    for _, row in bottom10.iterrows():
        true_str = "✓" if row['true_label'] == 1 else "✗"
        
        # ✅ FIXED: Handle NaN in indication
        indication = row.get('indication', '')
        if pd.isna(indication) or indication == '':
            indication = '(no indication provided)'
        else:
            indication = str(indication)[:50]
        
        print(f"  Rank {int(row['rank']):3d}: Score {row['triage_score']:5.1f} | {row['risk_level']:6s} | True: {true_str} | {indication}...")


def save_triage_results(results, mode):
    """Save triage results to CSV and summary text file."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    
    # Save full results
    csv_path = OUTPUT_DIR / f"triage_results_{mode}_{timestamp}.csv"
    results.to_csv(csv_path, index=False)
    print(f"\n💾 Full results saved to: {csv_path}")
    
    # Calculate AUCs for summary
    from sklearn.metrics import roc_auc_score
    
    mlp_auc = roc_auc_score(results['true_label'], results['mlp_prob']) if 'mlp_prob' in results else None
    cnn_auc = roc_auc_score(results['true_label'], results['cnn_prob']) if 'cnn_prob' in results else None
    ensemble_auc = roc_auc_score(results['true_label'], results['ensemble_prob']) if 'ensemble_prob' in results else None
    
    # Create summary dictionary with actual values
    summary = {
        'mode': mode,
        'timestamp': timestamp,
        'total_patients': len(results),
        'true_positives': int(results['true_label'].sum()),
        'true_negatives': int(len(results) - results['true_label'].sum()),
        'mean_score': f"{results['triage_score'].mean():.2f}",
        'median_score': f"{results['triage_score'].median():.2f}",
        'min_score': f"{results['triage_score'].min():.2f}",
        'max_score': f"{results['triage_score'].max():.2f}",
        'high_risk_count': int((results['risk_level'] == 'HIGH').sum()),
        'medium_risk_count': int((results['risk_level'] == 'MEDIUM').sum()),
        'low_risk_count': int((results['risk_level'] == 'LOW').sum()),
        'mlp_auc': f"{mlp_auc:.4f}" if mlp_auc else "N/A",
        'cnn_auc': f"{cnn_auc:.4f}" if cnn_auc else "N/A",
        'ensemble_auc': f"{ensemble_auc:.4f}" if ensemble_auc else "N/A"
    }
    
    # Save as readable text file
    summary_path = OUTPUT_DIR / f"triage_summary_{mode}_{timestamp}.txt"
    with open(summary_path, 'w') as f:
        f.write("="*60 + "\n")
        f.write(f"TRIAGE SUMMARY - {mode.upper()}\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Generated: {timestamp}\n")
        f.write(f"Total Patients: {summary['total_patients']}\n")
        f.write(f"True Positives (has condition): {summary['true_positives']}\n")
        f.write(f"True Negatives (no condition): {summary['true_negatives']}\n\n")
        
        f.write("SCORE DISTRIBUTION\n")
        f.write("-"*40 + "\n")
        f.write(f"Mean Score:   {summary['mean_score']}\n")
        f.write(f"Median Score: {summary['median_score']}\n")
        f.write(f"Min Score:    {summary['min_score']}\n")
        f.write(f"Max Score:    {summary['max_score']}\n\n")
        
        f.write("RISK LEVEL BREAKDOWN\n")
        f.write("-"*40 + "\n")
        f.write(f"HIGH Risk:   {summary['high_risk_count']} ({summary['high_risk_count']/summary['total_patients']*100:.1f}%)\n")
        f.write(f"MEDIUM Risk: {summary['medium_risk_count']} ({summary['medium_risk_count']/summary['total_patients']*100:.1f}%)\n")
        f.write(f"LOW Risk:    {summary['low_risk_count']} ({summary['low_risk_count']/summary['total_patients']*100:.1f}%)\n\n")
        
        f.write("MODEL PERFORMANCE (AUC)\n")
        f.write("-"*40 + "\n")
        f.write(f"MLP:      {summary['mlp_auc']}\n")
        f.write(f"CNN:      {summary['cnn_auc']}\n")
        f.write(f"Ensemble: {summary['ensemble_auc']}\n")
    
    print(f"💾 Summary saved to: {summary_path}")
    
    return csv_path, summary_path


def run_full_triage_pipeline(mode="pneumonia", image_dir=None):
    """
    Run the complete triage scoring pipeline.
    """
    # Generate triage scores
    results = generate_triage_report(mode=mode, image_dir=image_dir)
    
    # Print summary
    print_triage_summary(results, mode)
    
    # Save results
    output_path = save_triage_results(results, mode)
    
    return results, output_path


# ============================================================
# RUN THE PIPELINE
# ============================================================

if __name__ == "__main__":
    # Set your image directory
    image_dir = image_path
    
    # Run for pneumonia mode
    print("\n" + "="*80)
    print("TRIAGE SCORING SYSTEM")
    print("="*80)
    
    results_pneumonia, path_pneumonia = run_full_triage_pipeline(
        mode="pneumonia", 
        image_dir=image_dir
    )
    
    # Optional: Run for abnormal mode too
    run_abnormal = True # don't really have the domain knowledge required to back things up.  
    if run_abnormal:
        results_abnormal, path_abnormal = run_full_triage_pipeline(
            mode="abnormal",
            image_dir=image_dir
        )
    
    print("\n" + "="*80)
    print("✅ TRIAGE SCORING COMPLETE")
    print("="*80)


TRIAGE SCORING SYSTEM

LOADING MODELS FOR: PNEUMONIA
✅ Loaded test set: 573 patients
   Positives: 101 (17.6%)
✅ Loaded MLP (input dim: 1000)
✅ Loaded vectorizer
✅ Loaded CNN

GENERATING TRIAGE SCORES

🔮 Computing MLP predictions...
🖼️ Computing CNN predictions...

✅ Generated triage scores for 573 patients

TRIAGE SUMMARY - PNEUMONIA

📊 Score Distribution:
   Mean:   28.8
   Median: 23.5
   Min:    8.4
   Max:    86.3

🚦 Risk Level Breakdown:
   MEDIUM: 315 patients (55.0%)
   LOW: 188 patients (32.8%)
   HIGH: 70 patients (12.2%)

📈 Performance by True Label:

   Positive (Has Condition):
      Mean Triage Score: 46.9
      High Risk: 46 (45.5%)
      Medium Risk: 41 (40.6%)
      Low Risk: 14 (13.9%)

   Negative (No Condition):
      Mean Triage Score: 25.0
      High Risk: 24 (5.1%)
      Medium Risk: 274 (58.1%)
      Low Risk: 174 (36.9%)

🔴 TOP 10 HIGHEST RISK PATIENTS:
--------------------------------------------------------------------------------
  Rank  1: Score  86.3 | HI